# Pandas 第3课：数据清洗

> 业界常说数据分析 70% 时间在清洗数据。本课造一份典型"脏数据"：
> 有缺失、有重复、数字存成了文字、字符串带空格、还混着“缺”这种垃圾值。
>
> 参考：Pandas 官方 user guide、Machine Learning Plus 练习。

In [1]:
import pandas as pd
import numpy as np

dirty=pd.DataFrame({
    '姓名':[' 张三','李四 ','王五','张三','赵六'],
    '部门':['销售','技术',None,'销售','技术'],
    '销售额':['8000','12000','6500','8000','缺'],
    '年龄':[25,None,30,25,28]
})
print(dirty)
dirty.info()

    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   姓名      5 non-null      object 
 1   部门      4 non-null      object 
 2   销售额     5 non-null      object 
 3   年龄      4 non-null      float64
dtypes: float64(1), object(3)
memory usage: 288.0+ bytes


## 1. 缺失值检测

- `isna()` / `isnull()`（两者等价）：每个位置是否缺失
- `notna()`：是否非空
- `isna().sum()`：每列缺失个数（最常用）

In [ ]:
print(dirty.isna())
print(dirty.isna().sum())
print(dirty[dirty['部门'].notna()])

      姓名     部门    销售额     年龄
0  False  False  False  False
1  False  False  False   True
2  False   True  False  False
3  False  False  False  False
4  False  False  False  False
姓名     0
部门     1
销售额    0
年龄     1
dtype: int64
    姓名  部门    销售额    年龄
0   张三  销售   8000  25.0
1  李四   技术  12000   NaN
3   张三  销售   8000  25.0
4   赵六  技术      缺  28.0


## 2. 删除缺失：dropna

```python
df.dropna()              # 有任意缺失就删行（how='any'）
df.dropna(how='all')     # 整行全缺才删
df.dropna(subset=['部门'])# 只看某些列
df.dropna(thresh=3)      # 至少有3个非空值才保留
df.dropna(axis=1)        # 删列
```

In [3]:
print(dirty.dropna(subset=['部门']))
print(dirty.dropna(thresh=4))

    姓名  部门    销售额    年龄
0   张三  销售   8000  25.0
1  李四   技术  12000   NaN
3   张三  销售   8000  25.0
4   赵六  技术      缺  28.0
    姓名  部门   销售额    年龄
0   张三  销售  8000  25.0
3   张三  销售  8000  25.0
4   赵六  技术     缺  28.0


## 3. 填充缺失：fillna（优先于乱删）

```python
df.fillna(0)                       # 固定值
df['年龄'].fillna(df['年龄'].mean())# 均值/中位数填充
df.fillna(method='ffill')          # 用上一个值填充（forward）
df.fillna(method='bfill')          # 用下一个值
```
> 注意：method 参数新版 pandas 推荐直接写 `.ffill()` / `.bfill()`。

In [4]:
d=dirty.copy()
d['销售额'] = d['销售额'].fillna(0)
print(d)
d['年龄']=d['年龄'].fillna(d['年龄'].median())
d['部门']=d['部门'].fillna('未知')
print(d)

    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0
    姓名  部门    销售额    年龄
0   张三  销售   8000  25.0
1  李四   技术  12000  26.5
2   王五  未知   6500  30.0
3   张三  销售   8000  25.0
4   赵六  技术      缺  28.0


### 分组填充（更贴近真实：用同部门均值填）

In [5]:
d2=dirty.copy()
# 先把销售额转数字（见第5节），这里演示年龄按部门填
print("分组中位数思路：df.groupby('部门')['年龄'].transform('median')")

分组中位数思路：df.groupby('部门')['年龄'].transform('median')


## 4. 重复值：duplicated / drop_duplicates

- `duplicated()`：标记重复行（第一次出现不标记）
- `drop_duplicates()`：去重
  - `subset=[列]`：按某些列判断
  - `keep='first'/'last'/False`：保留哪个，False 表示全删

In [6]:
print(dirty.duplicated())
print(dirty.drop_duplicates())
print(dirty.drop_duplicates(subset=['姓名'], keep='last'))

0    False
1    False
2    False
3    False
4    False
dtype: bool
    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0
    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0


## 5. 类型转换（重点）

### to_numeric：把文字数字转回来，脏值变 NaN
```python
pd.to_numeric(s, errors='coerce')
# errors: raise(报错) / coerce(变NaN) / ignore(不动)
```
### to_datetime：转日期
### astype：明确转换

In [7]:
d=dirty.copy()
print(d)
d['销售额']=pd.to_numeric(d['销售额'], errors='coerce')
print(d)
d['销售额']=d['销售额'].fillna(0).astype(int)
print(d)

    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0
    姓名    部门      销售额    年龄
0   张三    销售   8000.0  25.0
1  李四     技术  12000.0   NaN
2   王五  None   6500.0  30.0
3   张三    销售   8000.0  25.0
4   赵六    技术      NaN  28.0
    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      0  28.0


### to_datetime 的 format 格式代码（解析 / 格式化日期）

`format` 用 `%` 开头的占位符描述日期长什么样，普通字符（年、月、日、横杠、斜杠）原样匹配。

- **字符串 → 日期**（to_datetime）：`pd.to_datetime(s, format='%Y年%m月%d日')`
- **日期 → 字符串**（反向，.dt.strftime）：`dt.dt.strftime('%d/%m/%Y')`

| 代码 | 含义 | 例子 |
|---|---|---|
| `%Y` | 四位年 | 2021 |
| `%y` | 两位年 | 21 |
| `%m` | 两位月 | 05 |
| `%d` | 两位日 | 01 |
| `%H` | 小时（24制） | 14 |
| `%M` | 分钟 | 30 |
| `%S` | 秒 | 00 |
| `%A` / `%a` | 星期全称 / 简称 | Monday / Mon |
| `%B` / `%b` | 月份全称 / 简称 | January / Jan |
| `%j` | 一年中第几天 | 121 |
| `%%` | 字面百分号 | % |

要点：
1. format 描述的是**数据现在的样子**，Pandas 照着模板读；模板里没写 `%` 的字符必须原样出现
2. 格式统一但特殊（如中文）用具体 format；横杠斜杠混用才用 `format='mixed'`
3. 指定 format 比自动推断**更快、更不会误判**，大文件和固定格式强烈建议写
4. strftime 是反向操作，把日期按模板拼成字符串，常用于做报表/导出展示


In [8]:
# 解析中文格式日期
s = pd.Series(['2021年05月01日','2021年12月25日'])
dt = pd.to_datetime(s, format='%Y年%m月%d日')
print("解析:", dt.tolist())
# 反向：日期拼成字符串
print("日/月/年:", dt.dt.strftime('%d/%m/%Y').tolist())
# 带时分秒
t = pd.to_datetime(pd.Series(['2021-05-01 14:30:00']), format='%Y-%m-%d %H:%M:%S')
print("带时间:", t.tolist())


解析: [Timestamp('2021-05-01 00:00:00'), Timestamp('2021-12-25 00:00:00')]
日/月/年: ['01/05/2021', '25/12/2021']
带时间: [Timestamp('2021-05-01 14:30:00')]


## 6. 替换：replace

- 按值替换，支持字典批量

In [9]:
d=dirty.copy()
print(d)
d['部门']=d['部门'].replace({None:'未知'})
print(d['部门'])
d['销售额']=d['销售额'].replace('缺', np.nan)
print(d['销售额'])

    姓名    部门    销售额    年龄
0   张三    销售   8000  25.0
1  李四     技术  12000   NaN
2   王五  None   6500  30.0
3   张三    销售   8000  25.0
4   赵六    技术      缺  28.0
0    销售
1    技术
2    未知
3    销售
4    技术
Name: 部门, dtype: object
0     8000
1    12000
2     6500
3     8000
4      NaN
Name: 销售额, dtype: object


### replace 深入：为什么用字典 + 正则模式清洗

**1. 为什么是键值对：字典是"对照表"，查找是手段、替换是目的**

字典本质是 `键→值` 的映射。replace 做的事：
- 拿元素当**键**去字典里**查**，得到对应的值
- 再用查到的值**替换**原元素
```
'男' ──查表──► 'M' ──写回──► 该位置变成 'M'
```
替换天然需要两个值（谁 → 变成谁），键值对正好表达"从旧到新"的对应，所以用字典。它和汇率表、单位换算表一个道理：原值查表 → 目标值 → 写回。字典的"查找"在这里是为"替换"服务的。

**2. 正则驱动 regex=True：按"模式"批量匹配，不用逐个枚举**

`replace(regex=模式, value=新值)`：
- `regex` 写正则表达式，描述"要找的字符长什么样"
- `value` 写替换成什么，写 `''` 等于删除
- 模式前的 `r` = 原始字符串，写正则习惯加上

| 正则 | 含义 |
|---|---|
| `[abc]` | a 或 b 或 c，任一字符 |
| `[0-9]` | 任意数字（范围） |
| `[a-zA-Z]` | 任意字母 |
| `[^0-9]` | 非数字（`^` 在 [] 里表示"非"） |
| `\d` / `\D` | 数字 / 非数字 |
| `\s` / `\w` | 空白 / 字母数字下划线 |
| `.` | 任意字符 |
| `+` `*` `?` | 前一模式出现 1次及以上 / 0次及以上 / 0或1次 |
| `^x` / `x$` | 开头 / 结尾 |

清洗最常用两条：
```python
s.replace(regex=r'[^\d.]', value='')   # 只留数字和小数点（提取数值）
s.replace(regex=r'\s+', value=' ')     # 连续空白压成一个
```

要点：
1. 字典 replace 适合"值可枚举"；regex 适合"值列不全、但有共同模式"
2. 正则提取出文本数字后，常接 `to_numeric` 转成真数字


In [10]:
import pandas as pd
s = pd.Series(['￥1,200','$2,300','3,400元'])
# 字符集合：删货币符号、逗号、单位
print(s.replace(regex=r'[￥$,元]', value='').tolist())
# 通用：只保留数字和小数点，再转数值
clean = s.replace(regex=r'[^\d.]', value='')
print(pd.to_numeric(clean).tolist())
# 连续空白压成一个
print(pd.Series(['张  三','李   四']).replace(regex=r'\s+', value=' ').tolist())


['1200', '2300', '3400']
[1200, 2300, 3400]
['张 三', '李 四']


## 7. 字符串清洗（Series.str 系列）

| 方法 | 作用 |
|---|---|
| `str.strip()` | 去首尾空格（还有 lstrip/rstrip） |
| `str.lower()/upper()` | 大小写 |
| `str.contains('x')` | 是否包含 |
| `str.replace(a,b)` | 替换 |
| `str.split(',')` | 切分（得到列表） |
| `str.len()` | 长度 |
| `str.startswith/endswith` | 开头/结尾 |

In [11]:
d=dirty.copy()
d['姓名']=d['姓名'].str.strip()
print(d['姓名'])
print(d[d['姓名'].str.contains('张')])

0    张三
1    李四
2    王五
3    张三
4    赵六
Name: 姓名, dtype: object
   姓名  部门   销售额    年龄
0  张三  销售  8000  25.0
3  张三  销售  8000  25.0


## 8. map / apply：自定义转换

- `Series.map(字典或函数)`：逐元素映射（最常用于把文字转数字/编码）
- `DataFrame.apply(函数, axis=1)`：对每行/每列套用
- 缺失映射：map 遇不到的键变 NaN

In [12]:
d=dirty.copy()
d['姓名']=d['姓名'].str.strip()
d['部门编码']=d['部门'].map({'销售':1,'技术':2})
print(d)
d['状态']=d['年龄'].apply(lambda x:'年轻' if (x is not None and x<28) else '其他')
print(d[['姓名','状态']])

   姓名    部门    销售额    年龄  部门编码
0  张三    销售   8000  25.0   1.0
1  李四    技术  12000   NaN   2.0
2  王五  None   6500  30.0   NaN
3  张三    销售   8000  25.0   1.0
4  赵六    技术      缺  28.0   2.0
   姓名  状态
0  张三  年轻
1  李四  其他
2  王五  其他
3  张三  年轻
4  赵六  其他


## 9. 异常值截断：clip

```python
df['销售额'].clip(lower=0, upper=100000)
```

## 10. 课堂综合练习

把开头的 `dirty` 清洗成一张干净表，要求：
1. 姓名去空格
2. 销售额转成整数，“缺”当作 0
3. 年龄缺失用整体中位数填充
4. 部门缺失填“未知”
5. 去掉重复行（按姓名，保留最后一条）
6. 最后打印 `info()` 确认无缺失、类型正确

In [11]:
# 在此完成（规范写法：先 copy，统一用赋值，不写"取出列再 inplace"）
d = dirty.copy()
d['姓名'] = d['姓名'].str.strip()
d['销售额'] = pd.to_numeric(d['销售额'], errors='coerce').fillna(0).astype(int)
d['年龄'] = d['年龄'].fillna(d['年龄'].median())
d['部门'] = d['部门'].fillna('未知')
d = d.drop_duplicates(subset=['姓名'], keep='last')
d.info()


<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 1 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   姓名      4 non-null      object 
 1   部门      4 non-null      object 
 2   销售额     4 non-null      int64  
 3   年龄      4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 160.0+ bytes


### 批改与知识点补充

**结果：6 项要求全部正确。** 销售额那条 `to_numeric(coerce).fillna(0).astype(int)` 链式尤其漂亮，顺序完全对（脏值→NaN→填0→转整数）。

**唯一要改的写法问题（对应你看到的 FutureWarning）：**
- 你写的 `dirty['年龄'].fillna(..., inplace=True)` 是"先取出列、再在这个中间对象上 inplace"。新版 Pandas 里这个中间对象可能是副本，inplace 改不到原表，Pandas 3.0 会彻底失效，所以报警告
- 改法：一律用赋值 `d['年龄'] = d['年龄'].fillna(...)`；另外练习先 `copy()`，别直接改原始 dirty

---

#### 1. inplace 是什么
- `inplace=False`（默认）：方法返回一个**新对象**，原对象不动
- `inplace=True`：直接改原对象、返回 None
- 趋势：官方逐渐弃用 inplace（尤其链式），**优先用赋值**，最稳

#### 2. `=` 和 `.` 的分工
- `.方法()`：负责"算"，返回结果，默认不改原对象
- `=`：负责"存"，把右边结果写进左边
- 固定搭配：`d['列'] = d['列'].方法()`，右边算、左边存
- 只有 `d['列']=值`、`d.loc[位置]=值` 这种赋值动作天生原地改

#### 3. 文字列里有 no/f/否 各种"脏写法"怎么办
- 写法有限：`replace({'no':'否','f':'否','N':'否'})` 一把统一
- 写法又多又乱：先 `.str.lower().str.strip()` 归一，再 replace；或正则匹配
- 最通用：`s.apply(函数)`，函数里 `if x in ['no','n','f','否','0']: ...`，任意可描述的规则都能兜底
- 你说得对，上游数据规范能省掉大量这种活，所以业界很强调采集规范；但处理历史/外部数据时仍得清洗

#### 4. 标准清洗流程（记住这条主线）
1. 备份 `d = df.copy()`
2. 看全貌：`head() / info() / describe()`
3. 字符串规整：`strip`、统一大小写、去多余符号
4. 类型转换：`to_numeric / to_datetime`（coerce 让脏值现形）
5. 缺失：`isna().sum()` 评估 → `fillna`（数值用均值/中位数，类别用众数/未知）或 `dropna`
6. 重复：`duplicated` → `drop_duplicates`
7. 异常值：describe/箱线发现 → `clip` 或核实
8. 文本归一/编码：`replace / map`
9. `info()` 确认 → `to_csv` 导出

#### 5. "这和 Excel 有什么区别"（你现在的疑问很正常）
基础动作确实像"可编程的 Excel"，但本质差别：
1. **规模**：Excel 几万行就卡，Pandas 处理百万千万行、GB 级
2. **可复现/自动化**：代码能一键重跑、定时执行、批量处理上百个文件，Excel 靠手点做不到
3. **能接下游**：Pandas 直接连数据库、爬虫、API、机器学习模型，Excel 是孤岛
4. **过程可审计**：代码即完整步骤，Excel 公式藏在格子里
- 你现在是在"攒零件"，所以像背 Excel 函数；等这些零件组装成自动化报表、喂给模型（MovieLens 项目），就是 Excel 永远做不到的东西

#### 6. 两个特殊输出
- **FutureWarning**：链式 inplace 的弃用提醒，按上面改赋值即可
- **memory usage: 160.0+ bytes**：160 是数值/索引可精确计算的部分；末尾 `+` 表示含 object（字符串）列，字符串散落在堆上、大小不一，没法精确计入


### 小结

- 缺失：`isna().sum()` 看情况，`dropna` 或 `fillna`（均值/前后填充）
- 重复：`drop_duplicates(subset,keep)`
- 类型：`to_numeric(errors='coerce')`、`to_datetime`、`astype`
- 字符串：`str.strip/contains/replace/split`
- 自定义：`map`（逐元素）、`apply`（逐行列）
- 清洗口诀：**先看缺失和类型，再去空格和重复，最后替换异常值**
- 下一课：groupby 分组聚合与多表合并